# 1. Answer the questions from the introduction

### 1.1 What is leave-one-out? Provide limitations and strengths.

### 1.2 How do Grid Search, Randomized Grid Search, and Bayesian optimization work?

### 1.3 Explain classification of feature selection methods.<br> Explain how Pearson and Chi2 work. Explain how Lasso works. <br> Explain what permutation significance is. Become familiar with SHAP.

# 2. Introduction — make all the preprocessing staff from the previous lesson

In [1]:
import warnings
import time
import datetime
from collections import Counter
from dataclasses import dataclass
from typing import Tuple, List
import pandas as pd
import numpy as np
from sklearn.preprocessing import (
    MultiLabelBinarizer,
    MinMaxScaler,
    StandardScaler,
    PolynomialFeatures,
)
from sklearn.model_selection import (
    train_test_split, 
    StratifiedKFold, 
    KFold)
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet,
    ElasticNetCV,
)
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import lightgbm as lgb
import scipy
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns


warnings.filterwarnings("ignore")

### 2.2 Read all the data

In [2]:
path: str = "data/two-sigma-connect-rental-listing-inquiries/{name}.json"
train_df: pd.DataFrame = pd.read_json(path.format(name="train"), convert_dates=['created'])
test_df: pd.DataFrame = pd.read_json(path.format(name="test"), convert_dates=['created'])

### 2.3 Preprocess the "Interest Level" feature.

In [3]:
values: dict = {"low": 0, "medium": 1, "high": 2}
train_df["interest_level"] = train_df["interest_level"].apply(lambda x: values[x])

### * Delete outliers

In [4]:
# more in chapter 11.2
lower_limit: np.float64 = train_df["price"].quantile(0.01)
upper_limit: np.float64 = train_df["price"].quantile(0.99)
train_df.drop(
    train_df[(train_df["price"] >= upper_limit) | \
             (train_df["price"] <= lower_limit)].index,
    inplace=True,
)
# for date split methods
train_col_created = train_df['created']

### 2.3 Create features: 'Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman', 'Dishwasher', 'NoFee', 'LaundryinBuilding', 'FitnessCenter', 'Pre-War', 'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'Balcony', 'SwimmingPool', 'LaundryInBuilding', 'NewConstruction', 'Terrace'.

In [5]:
feature_list: list = [
        "Elevator",
        "CatsAllowed",
        "HardwoodFloors",
        "DogsAllowed",
        "Doorman",
        "Dishwasher",
        "NoFee",
        "LaundryinBuilding",
        "FitnessCenter",
        "Pre-War",
        "LaundryinUnit",
        "RoofDeck",
        "OutdoorSpace",
        "DiningRoom",
        "HighSpeedInternet",
        "Balcony",
        "SwimmingPool",
        "LaundryInBuilding",
        "NewConstruction",
        "Terrace",
    ]

In [6]:
for df in train_df, test_df:
    mlb: MultiLabelBinarizer = MultiLabelBinarizer(classes=feature_list)
    new_features: pd.DataFrame = pd.DataFrame(
        mlb.fit_transform(df["features"]), index=df.index, columns=feature_list
    )
    df[feature_list] = new_features

new_feature_list: list = [*feature_list, "bathrooms", "bedrooms"]

X_train: pd.DataFrame = train_df[new_feature_list]
y_train: pd.DataFrame = train_df["price"]

X_test: pd.DataFrame = test_df[new_feature_list]
y_test: pd.DataFrame = test_df["price"]

display(X_train.head(3))
display(X_test.head(3))

,Elevator,CatsAllowed,HardwoodFloors,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,...,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace,bathrooms,bedrooms
4,0,0,0,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,1.0,1
6,1,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1.0,2
9,1,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1.0,2


,Elevator,CatsAllowed,HardwoodFloors,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,...,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace,bathrooms,bedrooms
0,1,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1.0,1
1,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1.0,2
2,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1.0,0


# 3. Implement the next methods:

### 3.5 Make split procedure determenistic. What does it mean?

Adding random_state to split makes random deterministic

### 3.1 Split data into 2 parts randomly with parameter test_size (ratio from 0 to 1), return training and test samples.

In [7]:
def check_split(df_list: List[pd.DataFrame], df_initial: pd.DataFrame) -> None:
    final_set: set = set(df_initial.index)
    for df in df_list:
        final_set -= set(df.index)
    print(final_set == set())
    [print(round(len(x) / len(df_initial), 2), end=' ') for x in df_list]

In [8]:
def simple_split(
        X: pd.DataFrame, 
        test_size: float = 0.2, 
        random_state: int = 42,
    ) -> Tuple[pd.DataFrame, pd.DataFrame]:
    
    X_test_: pd.DataFrame = X.sample(frac=test_size, random_state=random_state, replace=False)
    X_train_idx: list = list(set(X.index) - set(X_test_.index))
    X_train_: pd.DataFrame = X.loc[X_train_idx]
    return X_train_, X_test_


X_train_, X_test_ = simple_split(X=X_train, test_size=0.2)
check_split(df_list=[X_train_, X_test_], df_initial=X_train)

True
0.8 0.2 

### 3.2 Randomly split data into 3 parts with parameters validation_size and test_size, return train, validation and test samples.

In [9]:
def validation_split(
        X: pd.DataFrame, 
        test_size: float = 0.2, 
        validation_size: float = 0.2, 
        random_state: int = 42,
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    
    X_train_, X_test_ = simple_split(X, test_size=test_size, random_state=random_state)
    X_train_, X_valid_ = simple_split(X_train_, test_size=validation_size, random_state=random_state)
    return X_train_, X_valid_, X_test_


X_train_, X_valid_, X_test_ = validation_split(X=X_train, test_size=0.2, validation_size=0.25)
check_split(df_list=[X_train_, X_valid_, X_test_], df_initial=X_train)

True
0.6 0.2 0.2 

### 3.3 Split data into 2 parts with parameter date_split, return train and test samples split by date_split param.

In [10]:
X_train_with_date = X_train.copy()
X_train_with_date['date'] = train_col_created


def split_by_date(
        X: pd.DataFrame,
        date_split: datetime.datetime,
    ) -> Tuple[pd.DataFrame, pd.DataFrame] | Tuple:
    
    info: pd.Series = X.dtypes
    date_in_df = [pd.api.types.is_datetime64_ns_dtype(x) for x in  info.values]
    is_date_in_df = any(date_in_df)
    if not is_date_in_df:
        raise ValueError("No date col in df")
    
    date_col = info.index[date_in_df.index(True)]
    X_train_ = X.loc[X[date_col] <= date_split]
    X_test_ = X.loc[X[date_col] > date_split]


    time_condition = X_train_[date_col].max() < X_test_[date_col].max()
    if not time_condition:
        raise ValueError("Split date cant provide proper split")

    return X_train_, X_test_


split_by = datetime.datetime(year=2016, month=6, day=13)
X_train_, X_test_ = split_by_date(X=X_train_with_date, date_split=split_by)
check_split(df_list=[X_train_, X_test_], df_initial=X_train)


True
0.81 0.19 

### 3.4 Split data into 3 parts with parameters validation_date and test_date, return train, validation and test samples split by input params.

In [11]:
def validation_split_by_date(
        X: pd.DataFrame,
        test_date: datetime.datetime,
        validation_date: datetime.datetime,
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    
    X_train_, X_test_ = split_by_date(X, date_split=test_date)
    X_train_, X_valid_ = split_by_date(X_train_, date_split=validation_date)
    return X_train_, X_valid_, X_test_


test_date = datetime.datetime(year=2016, month=6, day=13)
validation_date = datetime.datetime(year=2016, month=5, day=25)
X_train_, X_valid_, X_test_ = validation_split_by_date(X=X_train_with_date, test_date=test_date, validation_date=validation_date)
check_split(df_list=[X_train_, X_valid_, X_test_], df_initial=X_train)


True
0.6 0.21 0.19 

# 4. Implement the next cross-validation methods:

### 4.1 K-Fold, where k is the input parameter, returns a list of train and test indices

<img src="data/KFold.png" width="25%" height="25%">

In [120]:
@dataclass
class KFold_:
    """KFold like in sklearn k = n_splits"""
    n_splits: int = 5
    shuffle: bool = True
    random_state: int = 42
    y: pd.DataFrame = None

    def split(self, X: pd.DataFrame, y: pd.DataFrame):
        """X - initial df"""
        n_samples: int = X.shape[0]
        if self.n_splits < 1:
            raise ValueError("Arg n_splits must be > 1")
        
        indexes = list(range(n_samples))

        if self.shuffle:
            rng: np.random.RandomState = np.random.RandomState(self.random_state)
            rng.shuffle(indexes)
        
        folds_list: list = []
        is_divisible: int | float = n_samples % self.n_splits
        test_len: int = n_samples // self.n_splits + 1
        pos: int = 0
        
        for _ in range(self.n_splits):
            if is_divisible:
                is_divisible -= 1
            else:
                test_len = n_samples // self.n_splits
            test_indexes = indexes[pos: pos + test_len]
            train_indexes = list(set(indexes) - set(test_indexes))

            folds_list.append([train_indexes, test_indexes])
            pos += test_len

        return folds_list

### 4.2 Grouped K-Fold, where k and group_field are input parameters, returns list of train and test indices.

<img src="data/GroupKFold.png" width="25%" height="25%">


In [ ]:
@dataclass
class GroupKFold_:
    """GroupKFold like in sklearn k = n_splits"""
    n_splits: int = 5
    shuffle: bool = True
    random_state: int = 42
    y: pd.DataFrame = None

    def split(self, X: pd.DataFrame, y: pd.DataFrame, groups:list = None):
        """X - initial df"""
        n_samples: int = X.shape[0]
        if self.n_splits < 1:
            raise ValueError("Arg n_splits must be > 1")
        
        if not groups or len(groups) != n_samples or all(groups):
            raise ValueError("Arg groups is empty / wrong len / has N/a")
        
        indexes = list(range(n_samples))

        if self.shuffle:
            rng: np.random.RandomState = np.random.RandomState(self.random_state)
            rng.shuffle(indexes)
        
        folds_list: list = []
        groups_keys = 

        # is_divisible: int | float = n_samples % self.n_splits
        # test_len: int = n_samples // self.n_splits + 1
        # pos: int = 0
        
        # for _ in range(self.n_splits):
        #     if is_divisible:
        #         is_divisible -= 1
        #     else:
        #         test_len = n_samples // self.n_splits
        #     test_indexes = indexes[pos: pos + test_len]
        #     train_indexes = list(set(indexes) - set(test_indexes))

        #     folds_list.append([train_indexes, test_indexes])
        #     pos += test_len

        return folds_list

In [148]:
import numpy as np
from sklearn.model_selection import GroupKFold
X = np.array([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10], [11, 12]])
y = np.array([1, 2, 3, 4, 5, 6])
groups = np.array([0, 0, 2, 2, 3, 3,])
group_kfold = GroupKFold(n_splits=2, shuffle=True, random_state=42)
group_kfold.get_n_splits(X, y, groups)
print(group_kfold)
for i, (train_index, test_index) in enumerate(group_kfold.split(X, y, groups)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}, group={groups[train_index]}")
    print(f"  Test:  index={test_index}, group={groups[test_index]}")

GroupKFold(n_splits=2, random_state=42, shuffle=True)
Fold 0:
  Train: index=[4 5], group=[3 3]
  Test:  index=[0 1 2 3], group=[0 0 2 2]
Fold 1:
  Train: index=[0 1 2 3], group=[0 0 2 2]
  Test:  index=[4 5], group=[3 3]


### 4.3 Stratified K-fold, where k and stratify_field are input parameters, returns list of train and test indices.

<img src="data/StratifiedKFold.png" width="25%" height="25%">


In [ ]:
X, y = np.ones((50, 1)), np.hstack(([0] * 45, [1] * 5))
skf = StratifiedKFold(n_splits=3)
# for train, test in skf.split(X, y):
#      print('train -  {}   |   test -  {}'.format(
#         np.bincount(y[train]), np.bincount(y[test])))

kf = KFold(n_splits=6, shuffle=True, random_state=42)
a = set(list(range(50)))
for train, test in kf.split(X, y):
    # print('train -  {}   |   test -  {}'.format(
    #     np.bincount(y[train]), np.bincount(y[test])))
    # print(len(train), train,'\n', len(test), test)
    print(len(test), test)
    a -= set(test)
    print(a)
a

In [ ]:
# def crossval(n_splits: int, X: pd.DataFrame, y: pd.DataFrame) -> str:
#     # skf = StratifiedKFold(n_splits=n_splits, random_state=21, shuffle=True)
#     skf = StratifiedKFold(n_splits=n_splits, shuffle=True)
#     out = ""; acc_list: list = []
#     # get split indexes 
#     for train_ix, test_ix in skf.split(X, y):
#         y_model = y.iloc[train_ix]; y_test = y.iloc[test_ix]
#         X_model = X.iloc[train_ix]; X_test = X.iloc[test_ix]
#         # X_train, X_valid, y_train, y_valid = train_test_split(X_model, y_model, test_size=0.25, random_state=21)

#         # model.fit(X_train, y_train)

#         # for x_split, y_split, text in ((X_train, y_train, 'train - {:.5f} | '), (X_valid, y_valid, 'valid - {:.5f}\n')):
#         #     predict_col = model.predict(x_split)
#         #     acc = accuracy_score(predict_col, y_split)
#         #     if text == 'valid - {:.5f}\n':
#         #         acc_list.append(acc)
#         #     out += text.format(acc)

#     # out += f'Average accuracy on crossval is {np.mean(acc_list):.5f}\n'
#     # out += f'Std is {np.std(acc_list):.5f}'
#         display(train_ix)
#     return out

# crossval(2, X_train, y_train)
# X_train.shape

### 4.4 Time series split, where k and date_field are input parameters, returns list of train and test indices.

<img src="data/TimeSeriesSplit.png" width="25%" height="25%">

# 5. Cross-validation comparison

### 5.1 Apply all the validation methods implemented above to our dataset. To apply Stratified algorithm you should preprocess target.

### 5.2 Apply the appropriate methods from sklearn.

### 5.3 Compare the resulting feature distributions for the training part of the dataset between sklearn and your implementation.

### 5.4 Compare all validation schemes. Choose the best one. Explain your choice.

# 6. Feature Selection

### 6.1 Fit a Lasso regression model with normalized features. Use your method for splitting samples into 3 parts by field created with 60/20/20 ratio — train/validation/test.

### 6.2 Sort features by weight coefficients from model, fit model to top 10 features and compare quality.

### 6.3 Implement method for simple feature selection by nan-ratio in feature and correlation. Apply this method to feature set and take top 10 features, refit model and measure quality.

### 6.4 Implement permutation importance method and take top 10 features, refit model and measure quality.

### 6.5 Import Shap and also refit model on top 10 features.

### 6.6 Compare the quality of these methods for different aspects — speed, metrics and stability.

# 7. Hyperparameter optimization

### 7.1 Implement grid search and random search methods for alpha and l1_ratio for sklearn's ElasticNet model.

### 7.2 Find the best combination of model hyperparameters.

### 7.3 Fit the resulting model.

### 7.4 Import optuna and configure the same experiment with ElasticNet.

### 7.5 Estimate metrics and compare approaches.

### 7.6 Run optuna on one of the cross-validation schemes.